# H-SegSplat — full pipeline on Colab

Runs the end-to-end H-SegSplat pipeline on a single 2-view scene:

1. Clone repo + download two checkpoints (SemanticSAM SwinL + DepthSplat).
2. Build three Python venvs (SAM, SigLIP, H-SegSplat). The CUDA op compile in stage 1 is the only fragile step; if it fails, see the troubleshooting notes at the bottom.
3. Upload one scene under `data/<scene>/dslr/...` (zipped).
4. Run `pipeline/run_pipeline.sh data/<scene>` — produces `data/<scene>/gaussians.pt`.
5. Download `gaussians.pt` for the local Unity viewer.

Each cell is idempotent. You can re-run the notebook safely on the same Colab instance.

**GPU runtime required.** Set `Runtime → Change runtime type → T4 GPU` (or better).

---

## 1. Clone repo and pick scene name

In [ ]:
# EDIT THIS: name of the scene directory you'll upload in step 4 below.
SCENE_NAME = 'my_scene'

# EDIT THIS: your fork or the upstream URL.
REPO_URL = 'https://github.com/0shean/h-segsplat.git'

import os, subprocess
os.chdir('/content')
if not os.path.isdir('/content/h-segsplat'):
    subprocess.check_call(['git', 'clone', REPO_URL, 'h-segsplat'])
os.chdir('/content/h-segsplat')
!ls

## 2. Download checkpoints

- **DepthSplat**: 3.6 GB, the pretrained dl3dv/RE10K checkpoint.
- **SemanticSAM**: ~2 GB, the SwinL many-to-many checkpoint.

Skipped if the files already exist.

In [ ]:
%cd /content/h-segsplat

# DepthSplat
!mkdir -p depthsplat/pretrained
DPT_CKPT='depthsplat/pretrained/depthsplat-gs-base-re10kdl3dv-448x768-randview2-6-f8ddd845.pth'
![ -f $DPT_CKPT ] || wget -q --show-progress -O $DPT_CKPT \
    https://huggingface.co/haofeixu/depthsplat/resolve/main/depthsplat-gs-base-re10kdl3dv-448x768-randview2-6-f8ddd845.pth

# SemanticSAM — replace URL with the official source if changed.
SAM_CKPT='swinl_only_sam_many2many.pth'
![ -f $SAM_CKPT ] || wget -q --show-progress -O $SAM_CKPT \
    https://github.com/UX-Decoder/Semantic-SAM/releases/download/checkpoint/swinl_only_sam_many2many.pth

!ls -lh depthsplat/pretrained/ swinl_only_sam_many2many.pth

## 3. Set up three venvs

Each setup cell takes 5–15 min the first time, instant on re-run (skipped if venv already exists).

**SemanticSAM venv** — installs detectron2 from source and compiles the `MultiScaleDeformableAttention` CUDA op.

In [ ]:
%cd /content/h-segsplat
import os
if not os.path.isdir('envs/sam/venv'):
    !bash envs/sam/setup.sh
else:
    print('envs/sam/venv exists — skipping setup')

In [ ]:
%cd /content/h-segsplat
import os
if not os.path.isdir('envs/siglip/venv'):
    !bash envs/siglip/setup.sh
else:
    print('envs/siglip/venv exists — skipping setup')

In [ ]:
%cd /content/h-segsplat
import os
if not os.path.isdir('envs/hsegsplat/venv'):
    !bash envs/hsegsplat/setup.sh
else:
    print('envs/hsegsplat/venv exists — skipping setup')

## 4. Upload scene data

Upload a zip containing your scene with this layout:

```
<scene_name>/
  dslr/
    nerfstudio/transforms.json
    resized_images/<frame>.JPG
```

The cell unzips into `data/<scene_name>/`.

In [ ]:
%cd /content/h-segsplat
from google.colab import files
import os, zipfile

if not os.path.isdir(f'data/{SCENE_NAME}/dslr/resized_images'):
    print(f'Upload your scene zip (it should contain a top-level {SCENE_NAME}/ directory)')
    up = files.upload()
    for fname in up:
        with zipfile.ZipFile(fname) as zf:
            zf.extractall('data/')
        os.remove(fname)

!ls -la data/$SCENE_NAME/dslr/resized_images/ | head
!cat data/$SCENE_NAME/dslr/nerfstudio/transforms.json | head -30

## 5. Run the pipeline

Runs all five stages in sequence. ~30–60 min on a T4 depending on number of frames.

In [ ]:
%cd /content/h-segsplat
!bash pipeline/run_pipeline.sh data/$SCENE_NAME

## 6. Download artifacts for the Unity viewer

The viewer needs `gaussians.pt`. The feature maps are optional (only needed for the 2D `query_hsegsplat.py` batch script).

In [ ]:
from google.colab import files
files.download(f'data/{SCENE_NAME}/gaussians.pt')
# Optional downloads:
# files.download(f'data/{SCENE_NAME}/rendered_rgb.npy')
# files.download(f'data/{SCENE_NAME}/rendered_feature_map_lvl1.npy')
# files.download(f'data/{SCENE_NAME}/rendered_feature_map_lvl3.npy')
# files.download(f'data/{SCENE_NAME}/rendered_feature_map_lvl6.npy')

## Troubleshooting

**`envs/sam/setup.sh` fails on the CUDA op build.** The compiled op (`MultiScaleDeformableAttention`) is bound to torch 2.1.2 / cu12.1. Colab's base CUDA version drifts. If Colab's nvcc is too old, the build fails. Workarounds:

1. Run stage 1 on the cluster instead — same scripts, run via `bash envs/sam/setup.sh` then `bash pipeline/stage_01_masks.sh data/<scene>`.
2. Pre-compile the `.so` somewhere and host it; this notebook would then download it instead of building.

**Out-of-memory in stage 5.** The DepthSplat encoder + three feature rasterizations needs ~12 GB GPU memory at 640×960 with 2 views. T4 is borderline. Try V100/A100 or downscale `dataset.image_shape` (you'd have to re-extract masks at the smaller resolution).